In [1]:
# At first i just import things that Hernik has done

In [ ]:
#smart import functions

import csv
import numpy as np

import sys, os
from pathlib import Path
sys.path.append(os.path.join(os.getcwd(), '..', '..'))
from utils.helpers import standardize, build_model_data, load_csv_data, read_csv_numeric
file_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FOLDER = os.path.join(file_path, '..', '..', 'dataset')

In [ ]:
x_indices, x_data = read_csv_numeric(os.path.join(DATA_FOLDER, "x_train.csv"))
y_indices, y_data = read_csv_numeric(os.path.join(DATA_FOLDER, "y_train.csv"))
del x_indices["Id"]
del y_indices["Id"]

In [ ]:
from utils.helpers import clean_data

x_data = clean_data(x_data, x_indices)

In [ ]:
interesting_columns = ["GENHLTH", "PHYSHLTH", "MENTHLTH", "POORHLTH", "HLTHPLN1", "MEDCOST", "CHECKUP1",
                        "BPHIGH4", "SEX", "WEIGHT2", "HEIGHT3", "QLACTLM2", 
                        "DIFFWALK", "SMOKE100", "SMOKDAY2", "EXERANY2", "EXRACT11"]

x_data = x_data[:, [x_indices[c] for c in interesting_columns]]

IndexError: index 35 is out of bounds for axis 1 with size 35

In [22]:
print("x_data.shape:", x_data.shape)
print("y_data.shape:", y_data.shape)
print(y_data)


x_data.shape: (328135, 35)
y_data.shape: (328135, 2)
[[ 0.00000e+00 -1.00000e+00]
 [ 1.00000e+00 -1.00000e+00]
 [ 2.00000e+00 -1.00000e+00]
 ...
 [ 3.28132e+05 -1.00000e+00]
 [ 3.28133e+05 -1.00000e+00]
 [ 3.28134e+05 -1.00000e+00]]


In [26]:
# CATEGORICAL OR CONTINUOUS?

# Divide features in x_train_clean into continuous and categorical

continuous_features = []
categorical_features = []

for idx, feature_name in enumerate(interesting_columns):
    name = feature_name
    col = x_data[:, idx]
    unique_vals = np.unique(col[~np.isnan(col)])
    # Heuristic: categorical if <= 10 unique values and all are integers
    if len(unique_vals) <= 20 and np.all(np.equal(np.mod(unique_vals, 1), 0)):
        categorical_features.append(name)
    else:
        continuous_features.append(name)

print("Continuous features:", continuous_features)
print("Categorical features:", categorical_features)

Continuous features: ['PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'WEIGHT2', 'HEIGHT3', 'EXRACT11', '_AGE80', '_BMI5']
Categorical features: ['GENHLTH', 'HLTHPLN1', 'MEDCOST', 'CHECKUP1', 'BPHIGH4', 'SEX', 'QLACTLM2', 'DIFFWALK', 'SMOKE100', 'SMOKDAY2', 'EXERANY2', 'HLTHPLN1', 'TOLDHI2', 'CHCCOPD1', 'CHCKIDNY', 'CHCOCNCR', 'CIMEMLOS', 'DIABETE3', 'EDUCA', 'EMPLOY1', 'WTCHSALT', 'HAREHAB1', 'SCNTMNY1', 'VETERAN3', 'INCOME2', '_LTASTH1', '_RACE']


In [ ]:
# BOXPLOTS

import os
n = x_data.shape[1]
# Create output folder for boxplots
boxplot_dir = "../eda_plots/boxplots"
os.makedirs(boxplot_dir, exist_ok=True)


for i, col_name in enumerate(interesting_columns):
    data = x_data[:, i]  # seleziona la colonna i-esima
    data = data.astype(float)

    # Filtriamo i NaN
    mask_not_nan = ~np.isnan(data)
    data_general = data[mask_not_nan]

    # Boxplot per y == -1 e y == 1
    data_neg1 = data[(y_data[:,1] == -1) & mask_not_nan]
    data_pos1 = data[(y_data[:,1] == 1) & mask_not_nan]

    # Evita colonne completamente NaN
    if len(data_general) == 0:
        continue

    plt.figure(figsize=(8, 6))
    plt.boxplot(
        [data_general, data_neg1, data_pos1],
        labels=["All", "y=-1", "y=1"],
        showfliers=False
    )
    plt.title(col_name)
    plt.ylabel("Value")
    plt.tight_layout()
    plt.savefig(os.path.join(boxplot_dir, f"{col_name}.png"))
    plt.close()

In [32]:
# FEATURE STATISTICS

import numpy as np
import csv
import matplotlib.pyplot as plt
import os

# --- Setup ---
feature_names = interesting_columns  # nomi delle colonne
x_train_clean = x_data               # dataset preprocessato
fs_dir = "../eda_plots/feature_stats"
os.makedirs(fs_dir, exist_ok=True)

# --- Calcola statistiche ---
stats = []
for i, name in enumerate(feature_names):
    col = x_train_clean[:, i].astype(float)
    n_total = col.shape[0]
    mask_notnan = ~np.isnan(col)
    col_nonan = col[mask_notnan]

    perc_nan = 100 * (n_total - col_nonan.size) / n_total  # percentuale di NaN

    if col_nonan.size == 0:  # evita colonne completamente NaN
        stats.append([name] + [np.nan]*8 + [perc_nan])
        continue

    mean = np.mean(col_nonan)
    median = np.median(col_nonan)
    var = np.var(col_nonan)
    q0 = np.percentile(col_nonan, 0)
    q1 = np.percentile(col_nonan, 25)
    q2 = np.percentile(col_nonan, 50)
    q3 = np.percentile(col_nonan, 75)
    q100 = np.percentile(col_nonan, 100)

    stats.append([name, mean, median, var, q0, q1, q2, q3, q100, perc_nan])

# --- Salva come CSV ---
header = ["Feature", "Mean", "Median", "Variance", "Min", "Q1", "Q2", "Q3", "Max", "PercNaN"]
with open("../eda_plots/feature_stats/feature_stats.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(stats)

# --- Salva come immagine ---
fig, ax = plt.subplots(figsize=(12, len(stats) * 0.3))
ax.axis("off")

table = ax.table(cellText=[[f"{v:.2f}" if isinstance(v,float) else v for v in row] for row in stats],
                 colLabels=header,
                 loc="center",
                 cellLoc="center")

table.auto_set_font_size(False)
table.set_fontsize(8)
table.auto_set_column_width(col=list(range(len(header))))

plt.tight_layout()
plt.savefig(os.path.join(fs_dir, "feature_stats.png"))
plt.close()


In [39]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Create output folder for histograms
hist_relfreq_dir = "../eda_plots/hists_relfreq"
os.makedirs(hist_relfreq_dir, exist_ok=True)

for i, col_name in enumerate(interesting_columns):
    col = x_data[:, i]
    mask_neg1 = (y_data[:,1] == -1) & (~np.isnan(col))
    mask_pos1 = (y_data[:,1] == 1) & (~np.isnan(col))
    data_neg1 = col[mask_neg1]
    data_pos1 = col[mask_pos1]
    fig, axes = plt.subplots(2, 1, figsize=(8, 10))
    # Decide bins
    if col_name in categorical_features:
        bins = np.unique(np.concatenate([data_neg1, data_pos1]))
        # Plot each category separately to allow gaps
        for val in bins:
            axes[0].bar(val, np.sum(data_neg1 == val) / len(data_neg1), 
                    width=1, color='blue', alpha=0.6, edgecolor='black', label='y=-1' if val == bins[0] else "")
            axes[0].bar(val, np.sum(data_pos1 == val) / len(data_pos1), 
                    width=1, color='red', alpha=0.6, edgecolor='black', label='y=1' if val == bins[0] else "")

        axes[0].set_xticks(bins)
        for val in bins:
            axes[1].bar(val, np.sum(data_neg1 == val), 
                    width=1, color='blue', alpha=0.6, edgecolor='black', label='y=-1' if val == bins[0] else "")
            axes[1].bar(val, np.sum(data_pos1 == val), 
                    width=1, color='red', alpha=0.6, edgecolor='black', label='y=1' if val == bins[0] else "")

        axes[1].set_xticks(bins)
    else:
        bins = 30
        # --- Relative frequency ---
        axes[0].hist(data_neg1, bins=bins, density=True, alpha=0.6, color='blue', label='y = -1', edgecolor='black')
        axes[0].hist(data_pos1, bins=bins, density=True, alpha=0.6, color='red', label='y = 1', edgecolor='black')
        

        # --- Absolute frequency ---
        axes[1].hist(data_neg1, bins=bins, density=False, alpha=0.6, color='blue', label='y = -1', edgecolor='black')
        axes[1].hist(data_pos1, bins=bins, density=False, alpha=0.6, color='red', label='y = 1', edgecolor='black')
        
    axes[0].set_title(f"Relative Frequency Histogram of {col_name} by y_train")
    axes[0].set_ylabel("Relative Frequency")
    axes[0].legend()
    axes[1].set_title(f"Absolute Frequency Histogram of {col_name} by y_train")
    axes[1].set_xlabel(col_name)
    axes[1].set_ylabel("Frequency")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(hist_relfreq_dir, f"{col_name}_hist.png"))
    plt.close()

C:\Users\andre\AppData\Local\Temp\ipykernel_29652\1170643506.py:21: RuntimeWarning: invalid value encountered in scalar divide
  axes[0].bar(val, np.sum(data_neg1 == val) / len(data_neg1),


In [41]:
# CORRELATION

import os

import matplotlib.pyplot as plt

corr_matrix = np.full((x_data.shape[1], x_data.shape[1]), np.nan)
for i in range(x_data.shape[1]):
    for j in range(x_data.shape[1]):
        col_i = x_data[:, i]
        col_j = x_data[:, j]
        mask = ~np.isnan(col_i) & ~np.isnan(col_j)
        if np.sum(mask) > 1 :
            std_i = np.std(col_i[mask])
            std_j = np.std(col_j[mask])
            if std_i == 0 or std_j == 0:
                corr_matrix[i,j] = np.nan
            else:
                corr_matrix[i,j] = np.corrcoef(col_i[mask], col_j[mask])[0,1]
np.fill_diagonal(corr_matrix, 1.0)

# Mark non-negligible correlations (absolute value > 0.05)
marked_005 = np.where(np.abs(corr_matrix) > 0.05, corr_matrix, np.nan)

# Plot and save
heatmap_dir = "../eda_plots/heatmaps"
os.makedirs(heatmap_dir, exist_ok=True)

# Plot and save for |corr| > 0.05
plt.figure(figsize=(14, 12))
im = plt.imshow(marked_005, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Correlation Matrix (|corr| > 0.05 marked)")
plt.xticks(range(len(interesting_columns)), interesting_columns, fontsize=8)
plt.yticks(range(len(interesting_columns)), interesting_columns, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(heatmap_dir, "correlation_matrix_marked_005.png"))
plt.close()

In [ ]:
# save corrs in tables

import os
import csv
import numpy as np
import matplotlib.pyplot as plt

# Ensure directory exists
heatmap_dir = "../eda_plots/heatmaps"
os.makedirs(heatmap_dir, exist_ok=True)

threshold = 0.05
correlation_list = []

# --- Collect all pairs with |corr| > 0.05 (excluding diagonal and duplicates) ---
for i in range(len(interesting_columns)):
    for j in range(i + 1, len(interesting_columns)):
        val = corr_matrix[i, j]
        if not np.isnan(val) and abs(val) > threshold:
            correlation_list.append([
                interesting_columns[i],
                interesting_columns[j],
                round(val, 4)
            ])

# --- Sort by descending absolute correlation ---
correlation_list.sort(key=lambda x: abs(x[2]), reverse=True)

# --- Save as CSV (all correlations) ---
csv_path = os.path.join(heatmap_dir, "correlations_marked_005.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Feature 1", "Feature 2", "Correlation"])
    writer.writerows(correlation_list)

print(f"✅ CSV table saved to: {csv_path}")

# --- Split into chunks of 40 correlations per image ---
chunk_size = 40
if correlation_list:
    num_chunks = int(np.ceil(len(correlation_list) / chunk_size))
    for k in range(num_chunks):
        start = k * chunk_size
        end = min((k + 1) * chunk_size, len(correlation_list))
        chunk = correlation_list[start:end]

        fig, ax = plt.subplots(figsize=(10, 0.35 * len(chunk) + 2))
        ax.axis("off")

        # Build table data
        table_data = [["Feature 1", "Feature 2", "Correlation"]] + chunk
        table = ax.table(cellText=table_data, loc="center", cellLoc="center")

        # Styling
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.2, 1.2)

        # Header style
        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold", color="white")
                cell.set_facecolor("#4B6A9B")
            elif col == 2:
                val = table_data[row][col]
                if isinstance(val, (float, int)):
                    color = plt.cm.coolwarm((val + 1) / 2)
                    cell.set_facecolor(color)
                    cell.set_text_props(color="black")

        plt.title(f"Feature Correlations |corr| > {threshold}  (Part {k+1}/{num_chunks})", pad=15)

        img_path = os.path.join(heatmap_dir, f"correlations_marked_005_part{k+1}.png")
        plt.savefig(img_path, bbox_inches="tight", dpi=200)
        plt.close()

        print(f"✅ Image table saved to: {img_path}")
else:
    print("⚠️ No correlations above threshold found.")


✅ CSV table saved to: ../eda_plots/heatmaps\correlations_marked_005.csv
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part1.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part2.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part3.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part4.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part5.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part6.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part7.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part8.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part9.png
✅ Image table saved to: ../eda_plots/heatmaps\correlations_marked_005_part10.png


In [47]:
# SCATTERPLOTS

import os
import numpy as np
import matplotlib.pyplot as plt

# --- Setup output directory ---
scatter_dir = "../eda_plots/scatterplots"
os.makedirs(scatter_dir, exist_ok=True)

# --- Parameters ---
features_per_group = 4  # number of features per subgroup
continuous_subgroups = [
    continuous_features[i:i + features_per_group]
    for i in range(0, len(continuous_features), features_per_group)
]

# --- Loop through subgroups ---
for g_idx, subgroup in enumerate(continuous_subgroups):
    num_feats = len(subgroup)
    if num_feats < 2:
        continue  # need at least 2 features for scatterplots

    # Determine number of subplot rows and columns
    n_pairs = num_feats * (num_feats - 1) // 2
    n_cols = min(3, n_pairs)  # max 3 columns per row
    n_rows = int(np.ceil(n_pairs / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.array(axes).reshape(-1)  # flatten in case of single row/col

    pair_idx = 0
    for i in range(num_feats):
        for j in range(i + 1, num_feats):
            f1 = subgroup[i]
            f2 = subgroup[j]

            idx1 = interesting_columns.index(f1)
            idx2 = interesting_columns.index(f2)

            x1 = x_data[:, idx1]
            x2 = x_data[:, idx2]

            # Remove NaNs
            mask = ~np.isnan(x1) & ~np.isnan(x2)
            x1_clean = x1[mask]
            x2_clean = x2[mask]
            y_clean = y_data[mask, 1]

            ax = axes[pair_idx]
            ax.scatter(x1_clean[y_clean == -1], x2_clean[y_clean == -1],
                       c='blue', alpha=0.5, s=15, marker='o', label='y = -1')
            ax.scatter(x1_clean[y_clean == 1], x2_clean[y_clean == 1],
                       c='red', alpha=0.5, s=15, marker='^', label='y = 1')
            ax.set_xlabel(f1)
            ax.set_ylabel(f2)
            ax.set_title(f"{f1} vs {f2}")
            ax.legend(fontsize=8)
            pair_idx += 1

    # Hide any unused subplots
    for k in range(pair_idx, len(axes)):
        axes[k].axis('off')

    plt.tight_layout()
    fname = f"group{g_idx+1}_scatter.png"
    plt.savefig(os.path.join(scatter_dir, fname))
    plt.close()


C:\Users\andre\AppData\Local\Temp\ipykernel_29652\3747558122.py:67: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(os.path.join(scatter_dir, fname))


In [48]:
print(continuous_subgroups)

[['PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'WEIGHT2'], ['HEIGHT3', 'EXRACT11', '_AGE80', '_BMI5']]
